# TrashID - Image Classification for Waste Management

Notebook ini dibuat untuk melatih model klasifikasi gambar sampah (Organik, Anorganik, dan Residu) menggunakan pendekatan **Transfer Learning**.

### Mengapa menggunakan MobileNetV2?
Dari beberapa pilihan arsitektur (ResNet50, MobileNetV2, EfficientNet), kita memilih **MobileNetV2**. Alasan utamanya adalah:
1. **Ukuran Ringan & Optimal untuk CPU**: Sangat cocok jika resources terbatas atau deployment dilakukan di server standar/perangkat mobile.
2. **Akurasi yang Baik**: MobileNetV2 memiliki profil ekstraksi fitur yang cukup kaya dan cenderung memiliki tingkat 
   *overfitting* yang lebih bisa dikontrol dengan *dropout* dan augmentasi untuk dataset yang tidak terlalu masif dibandingkan model jumbo.
3. **Kecepatan Inferensi Cepat**: Inferensi akan sangat responsif saat model digunakan langsung di aplikasi.

## 1. Import Library

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

# Set seeds untuk reproduksibilitas
np.random.seed(42)
tf.random.set_seed(42)

AttributeError: Module 'scipy' has no attribute '_lib'

## 2. Data Preprocessing & Augmentation

Sesuai struktur pada _workspace_, dataset tersusun di dalam `../data/Dataset_WasteWise_Final` dengan folder `train`, `val`, dan `test`. Kita menggunakan `ImageDataGenerator` untuk melakukan augmentasi pada data latih guna mencegah overfitting.

In [ ]:
base_dir = '../data/Dataset_WasteWise_Final'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

IMG_SIZE = (224, 224)
BATCH_SIZE = 32 # Gunakan batch lebih kecil (misal 16) bila RAM terbatas

# Data Augmentation untuk melatih model agar lebih adaptif
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())
print("Classes:", class_names)

## 3. Arsitektur Model (Transfer Learning)

Kita akan menggunakan MobileNetV2 sebagai *base model*, lalu membekukan layer aslinya dan menambahkan classifier (klasifikasi 3 kelas) dengan layer `Dropout` untuk mencegah *overfitting*.

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Bekukan base model agar weights aslinya tidak rusak saat awal proses train
base_model.trainable = False 

# Tambahkan custom classifier layer
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x) # Dropout mencegah overfitting
predictions = Dense(3, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 4. Pelatihan Model
Kita menset *epoch* maksimal misalkan 50. Namun, durasi asli akan ditentukan secara otomatis oleh `EarlyStopping` bergantung pada performa validasi loss. Ini teknik paling efisien untuk melatih tanpa rentan *overfitting*. 

In [ ]:
# Simpan model terbaik selama train
checkpoint = ModelCheckpoint(
    '../models/model_trashid.keras', 
    monitor='val_accuracy', 
    save_best_only=True, 
    save_weights_only=False, 
    verbose=1
)

# Berhenti otomatis jika tak ada peningkatan selama 7 epoch berturut-turut
early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=7, 
    restore_best_weights=True, 
    verbose=1
)

# Turunkan learning rate saat validation loss tidak kunjung membaik
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2, 
    patience=3,
    min_lr=1e-6,
    verbose=1
)

EPOCHS = 50

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stopping, reduce_lr]
)

## 5. Visualisasi Hasil Training
Melihat kurva Accuracy & Loss selama pergerakan epoch.

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

## 6. Evaluasi dan Pengujian (Test Data)
Kita memprediksi data dari folder `test` untuk mendapatkan parameter evaluasi yang *unbiased*.

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print(f'\nTest Accuracy: {test_acc:.4f}')

Y_pred = model.predict(test_generator)
y_pred = np.argmax(Y_pred, axis=1)

cm = confusion_matrix(test_generator.classes, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.title('Confusion Matrix')
plt.show()

print('\nClassification Report')
print(classification_report(test_generator.classes, y_pred, target_names=class_names))

## 7. Inferensi (Prediksi Gambar Baru)
Bagaimana cara memanggil model untuk mengklasifikasi satu buah gambar.

In [ ]:
from tensorflow.keras.preprocessing import image

def classify_image(img_path):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) # Tambahkan dimensi batch
    img_array = img_array / 255.0 # Skala yang sama dengan saat training
    
    prediction = model.predict(img_array)
    predicted_class_idx = np.argmax(prediction[0])
    predicted_class = class_names[predicted_class_idx]
    confidence = prediction[0][predicted_class_idx] * 100
    
    plt.imshow(img)
    plt.title(f'Prediksi: {predicted_class} ({confidence:.2f}%)')
    plt.axis('off')
    plt.show()
    
    return predicted_class, confidence

# Contoh penggunaan (Pastikan path mengarah ke file gambar yang valid)
# contoh_gambar = '../data/Dataset_WasteWise_Final/test/Organik/O (1).jfif'
# hasil, konfidensi = classify_image(contoh_gambar)